In [27]:
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import warnings
import json

# Suppress warnings
warnings.simplefilter(action='ignore', category=Warning)

fsize = 15
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'

In [28]:
def load_evidence(fn, ds_name):
    # we only want derived since we're going to compare gene ids
    df = pd.read_json(fn)
    ddf = df['derived'].apply(pd.Series)
    ddf["ds_name"] = ds_name

    # normalize the string values for cell types
    ddf["cell_type_label"] = ddf["cell_type_label"].str.strip().str.upper()
    ddf["cell_type_id"] = ddf["cell_type_id"].str.strip().str.upper()
    
    # normalize the string values for genes
    ddf["feature_name"] = ddf["feature_name"].str.strip().str.upper()
    ddf["feature_identifier"] = ddf["feature_identifier"].str.strip().str.upper()
    hddf = ddf.query("organism == 'homo_sapiens'")
    return hddf

def get_ctmap(ctmap_fn):
    with open(ctmap_fn, 'r') as file:
        map_dict = json.load(file)
        upper_map = {k.upper(): [v.upper() for v in vals] for k, vals in map_dict.items()}
        rev_map = {v: k for k, vals in upper_map.items() for v in vals}
    return (upper_map, rev_map)

In [29]:
fns = glob.glob("../../data/*")

In [30]:
hmns = []
degs = []
llms = []

gmap_fn = "../../data/gmap.txt"
gmap = (pd.read_csv(gmap_fn, header=None, sep="\t", names=["gname", "ensembl_id"])
    .drop_duplicates("gname")
    .applymap(lambda x: x.strip().upper() if isinstance(x, str) else x)
).set_index("gname")["ensembl_id"].to_dict()

for fn in fns:
    ds_name = fn.split("/")[-1]
    hmn_fn = os.path.join(fn, "evidence_human/evidence.json")
    deg_fn = os.path.join(fn, "evidence_deg/evidence.json")
    llm_fn = glob.glob(os.path.join(fn, "evidence_llm*/evidence.json"))
    llm_fn = llm_fn[0] if len(llm_fn) > 0 else None
    ctmap_fn = os.path.join(fn, "ctmap/ctmap.json")


    if os.path.exists(ctmap_fn):
        ctmap, rev_ctmap = get_ctmap(ctmap_fn)
        if os.path.exists(hmn_fn):
            hmn = load_evidence(hmn_fn, ds_name)
            hmn["cell_type_id_mapped"] = hmn["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            hmn["found_cell_type_id"] = hmn["cell_type_id"] == hmn["cell_type_id_mapped"]

            hmn["feature_id_mapped"] = hmn["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            hmn["found_feature_id"] = hmn["feature_identifier"] == hmn["feature_id_mapped"]

            hmns.append(hmn)

        if os.path.exists(deg_fn):
            deg = load_evidence(deg_fn, ds_name)
            deg["cell_type_id_mapped"] = deg["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            deg["found_cell_type_id"] = deg["cell_type_id"] == deg["cell_type_id_mapped"]

            deg["feature_id_mapped"] = deg["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            deg["found_feature_id"] = deg["feature_identifier"] == deg["feature_id_mapped"]

            degs.append(deg)
        if os.path.exists(llm_fn):
            llm = load_evidence(llm_fn, ds_name)
            llm["cell_type_id_mapped"] = llm["cell_type_label"].map(lambda x: rev_ctmap.get(x.strip().upper(), None))
            llm["found_cell_type_id"] = llm["cell_type_id"] == llm["cell_type_id_mapped"]

            llm["feature_id_mapped"] = llm["feature_name"].map(lambda x: gmap.get(x.strip().upper(), None))
            llm["found_feature_id"] = llm["feature_identifier"] == llm["feature_id_mapped"]

            llms.append(llm)

hmn = pd.concat(hmns)
deg = pd.concat(degs)
llm = pd.concat(llms)

# Human annotations

In [31]:
agg = hmn.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [32]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id          \
                                                    complete   found   
0          adipose_Emont2022        False               True   373.0   
1       adipose_Hildreth2021        False               True   170.0   
2         adipose_Jaitin2019         True               True    11.0   
3        adipose_Massier2023        False              False   197.0   
4        adipose_Merrick2019        False               True    29.0   
5          adipose_Vijay2019        False              False    92.0   
6             bladder_Yu2019        False               True    79.0   
7                bone_He2021        False               True   399.0   
8             eye_Gautam2021        False               True   373.0   
9           heart_Tucker2020        False               True   559.0   
10             kidney_Wu2018        False               True   259.0   
11        liver_Guillams2022        False               True    74.0   
12            lung_Adams2020        False               True   138.0   
13          ovary_Wagner2020        False               True   145.0   
14  pancreas_Segerstolpe2016        False               True   136.0   
15          placenta_Liu2018        False               True   200.0   
16         testis_Shamis2020        False               True   182.0   
17           yolksac_Goh2023        False               True   233.0   
18                     total        False              False  3649.0   

                       found_feature_id                              
           pct   total         complete   found         pct   total  
0   100.000000   373.0            False   367.0   98.391421   373.0  
1   100.000000   170.0            False   160.0   94.117647   170.0  
2   100.000000    11.0             True    11.0  100.000000    11.0  
3    76.356589   258.0            False   251.0   97.286822   258.0  
4   100.000000    29.0            False    21.0   72.413793    29.0  
5    87.619048   105.0            False   103.0   98.095238   105.0  
6   100.000000    79.0            False    75.0   94.936709    79.0  
7   100.000000   399.0            False   385.0   96.491228   399.0  
8   100.000000   373.0            False   352.0   94.369973   373.0  
9   100.000000   559.0            False   531.0   94.991055   559.0  
10  100.000000   259.0            False   246.0   94.980695   259.0  
11  100.000000    74.0            False    68.0   91.891892    74.0  
12  100.000000   138.0            False   135.0   97.826087   138.0  
13  100.000000   145.0            False   142.0   97.931034   145.0  
14  100.000000   136.0            False   133.0   97.794118   136.0  
15  100.000000   200.0            False   187.0   93.500000   200.0  
16  100.000000   182.0            False   177.0   97.252747   182.0  
17  100.000000   233.0            False   217.0   93.133047   233.0  
18   98.012356  3723.0            False  3561.0   95.648670  3723.0

In [33]:
# find specific datasets with missing cell type or feature name
ds = 'ovary_Wagner2020'
hmn.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]]

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
4,OOGONIAL STEM CELLS (OSCS),PROGENITOR CELLS,PROGENITOR CELLS,DDX4-AB,ENSG00000152670,None
8,IMMUNE CELLS,IMMUNE CELLS,IMMUNE CELLS,CD3G,ENSG00000289746,ENSG00000160654
43,FETAL GERMLINE CELLS (FGCS),GERM CELLS,GERM CELLS,STAR8,ENSG00000147465,None


# DEG annotations

In [34]:
agg = deg.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [35]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id          \
                                                    complete   found   
0          adipose_Emont2022        False               True   347.0   
1       adipose_Hildreth2021        False               True   220.0   
2         adipose_Jaitin2019        False               True   160.0   
3        adipose_Massier2023        False              False    58.0   
4          adipose_Vijay2019        False              False    47.0   
5             bladder_Yu2019        False               True    67.0   
6                bone_He2021        False               True   372.0   
7             eye_Gautam2021        False               True   420.0   
8           heart_Tucker2020        False               True   258.0   
9              kidney_Wu2018        False               True   308.0   
10        liver_Guillams2022        False               True   340.0   
11            lung_Adams2020        False               True    48.0   
12          ovary_Wagner2020        False               True   111.0   
13  pancreas_Segerstolpe2016        False               True   130.0   
14          placenta_Liu2018        False               True   103.0   
15         testis_Shamis2020        False               True   199.0   
16           yolksac_Goh2023        False               True   277.0   
17                     total        False              False  3465.0   

                       found_feature_id                             
           pct   total         complete   found        pct   total  
0   100.000000   347.0            False   314.0  90.489914   347.0  
1   100.000000   220.0            False   214.0  97.272727   220.0  
2   100.000000   160.0            False   158.0  98.750000   160.0  
3    93.548387    62.0            False    61.0  98.387097    62.0  
4    88.679245    53.0            False    49.0  92.452830    53.0  
5   100.000000    67.0            False    64.0  95.522388    67.0  
6   100.000000   372.0            False   356.0  95.698925   372.0  
7   100.000000   420.0            False   396.0  94.285714   420.0  
8   100.000000   258.0            False   250.0  96.899225   258.0  
9   100.000000   308.0            False   291.0  94.480519   308.0  
10  100.000000   340.0            False   323.0  95.000000   340.0  
11  100.000000    48.0            False    44.0  91.666667    48.0  
12  100.000000   111.0            False   103.0  92.792793   111.0  
13  100.000000   130.0            False   120.0  92.307692   130.0  
14  100.000000   103.0            False    95.0  92.233010   103.0  
15  100.000000   199.0            False   188.0  94.472362   199.0  
16  100.000000   277.0            False   269.0  97.111913   277.0  
17   99.712230  3475.0            False  3295.0  94.820144  3475.0

In [36]:
# find specific datasets with missing cell type or feature name
ds = 'bone_He2021'
deg.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]].head(3)

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
17,OCP,PROGENITOR CELLS,PROGENITOR CELLS,OSR1,ENSG00000172939,ENSG00000143867
68,MAROPHAGE,MACROPHAGE,MACROPHAGE,ELF1,ENSG00000130165,ENSG00000120690
80,MAROPHAGE,MACROPHAGE,MACROPHAGE,LAT2,ENSG00000092068,ENSG00000086730


# LLM annotations

In [37]:
agg = hmn.groupby("ds_name").agg({
    "found_feature_id": ["count", "sum"], 
    "found_cell_type_id": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_id", "pct")] = 100*agg[("found_feature_id", "found")] / agg[("found_feature_id", "total")]
agg[("found_cell_type_id", "pct")] = 100*agg[("found_cell_type_id", "found")] / agg[("found_cell_type_id", "total")]
agg[("found_feature_id", "complete")] = agg[("found_feature_id", "pct")] == 100
agg[("found_cell_type_id", "complete")] = agg[("found_cell_type_id", "pct")] == 100
agg["all_complete"] = agg[("found_feature_id", "complete")] & agg[("found_cell_type_id", "complete")]

In [38]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_id", "pct")] = 100 * total_row[("found_feature_id", "found")] / total_row[("found_feature_id", "total")]
total_row[("found_cell_type_id", "pct")] = 100 * total_row[("found_cell_type_id", "found")] / total_row[("found_cell_type_id", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_id", "complete")] = total_row[("found_feature_id", "pct")] == 100
total_row[("found_cell_type_id", "complete")] = total_row[("found_cell_type_id", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_id", "complete")] & total_row[("found_cell_type_id", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

ds_name all_complete found_cell_type_id          \
                                                    complete   found   
0          adipose_Emont2022        False               True   373.0   
1       adipose_Hildreth2021        False               True   170.0   
2         adipose_Jaitin2019         True               True    11.0   
3        adipose_Massier2023        False              False   197.0   
4        adipose_Merrick2019        False               True    29.0   
5          adipose_Vijay2019        False              False    92.0   
6             bladder_Yu2019        False               True    79.0   
7                bone_He2021        False               True   399.0   
8             eye_Gautam2021        False               True   373.0   
9           heart_Tucker2020        False               True   559.0   
10             kidney_Wu2018        False               True   259.0   
11        liver_Guillams2022        False               True    74.0   
12            lung_Adams2020        False               True   138.0   
13          ovary_Wagner2020        False               True   145.0   
14  pancreas_Segerstolpe2016        False               True   136.0   
15          placenta_Liu2018        False               True   200.0   
16         testis_Shamis2020        False               True   182.0   
17           yolksac_Goh2023        False               True   233.0   
18                     total        False              False  3649.0   

                       found_feature_id                              
           pct   total         complete   found         pct   total  
0   100.000000   373.0            False   367.0   98.391421   373.0  
1   100.000000   170.0            False   160.0   94.117647   170.0  
2   100.000000    11.0             True    11.0  100.000000    11.0  
3    76.356589   258.0            False   251.0   97.286822   258.0  
4   100.000000    29.0            False    21.0   72.413793    29.0  
5    87.619048   105.0            False   103.0   98.095238   105.0  
6   100.000000    79.0            False    75.0   94.936709    79.0  
7   100.000000   399.0            False   385.0   96.491228   399.0  
8   100.000000   373.0            False   352.0   94.369973   373.0  
9   100.000000   559.0            False   531.0   94.991055   559.0  
10  100.000000   259.0            False   246.0   94.980695   259.0  
11  100.000000    74.0            False    68.0   91.891892    74.0  
12  100.000000   138.0            False   135.0   97.826087   138.0  
13  100.000000   145.0            False   142.0   97.931034   145.0  
14  100.000000   136.0            False   133.0   97.794118   136.0  
15  100.000000   200.0            False   187.0   93.500000   200.0  
16  100.000000   182.0            False   177.0   97.252747   182.0  
17  100.000000   233.0            False   217.0   93.133047   233.0  
18   98.012356  3723.0            False  3561.0   95.648670  3723.0

In [39]:
# find specific datasets with missing cell type or feature name
ds = 'adipose_Massier2023'
llm.query(
    f"ds_name == '{ds}' & (~found_cell_type_id | ~found_feature_id)"
)[["cell_type_label", "cell_type_id",  "cell_type_id_mapped", "feature_name", "feature_identifier",  "feature_id_mapped"]].head(3)

,cell_type_label,cell_type_id,cell_type_id_mapped,feature_name,feature_identifier,feature_id_mapped
0,TH1-POLARIZED,T CELL,T CELL,LYC0,None,None
1,TISSUE-RESIDENT MEMORY (TRM),TISSUE-RESIDENT MEMORY (TRM),None,LYC01,None,None
2,NAIVE/EARLY DIFFERENTIATED,NAIVE/EARLY DIFFERENTIATED,None,LYC04,None,None
